# Self-Supervised Bioimage Denoising

## Learning from Noisy Microscopy Images Without Clean Targets


## Overview

Biological imaging often involves a difficult trade-off.

A researcher may want to observe living cells over a long period of time. Increasing the imaging exposure can produce clearer images, but repeated or intense illumination may increase photobleaching and phototoxicity.

Reducing the illumination can make long-term imaging safer for the biological sample, but the resulting microscopy images may contain much more noise.

This creates a practical problem:

> **Can we recover useful biological structures from noisy microscopy images when clean ground-truth images are unavailable?**

Traditional supervised denoising assumes that each noisy input image has a corresponding clean target:

$$x_{\mathrm{noisy}} \rightarrow f_\theta \rightarrow x_{\mathrm{clean}}$$

However, obtaining clean targets in biological imaging can be difficult.

A cleaner image may require longer exposure, repeated acquisition, or stronger illumination. For dynamic samples such as living cells, the biological scene may also change between acquisitions.

Self-supervised denoising provides another possibility.

Instead of training with clean targets, the model learns directly from noisy observations.

This tutorial introduces the ideas behind two influential approaches:

* **Noise2Void**
* **Noise2Self**

Noise2Void demonstrates how a denoiser can learn from noisy images alone by hiding selected pixels and predicting them from their surrounding context.

Noise2Self provides a more general theoretical framework for understanding why this type of self-supervision can work.

Rather than training a large production-scale microscopy restoration model, this notebook focuses on the core ideas and computational mechanisms behind self-supervised denoising.

---

## Core Scenario

Imagine that you are studying living cells with fluorescence microscopy.

You want to observe the sample for a long period of time.

A high-exposure acquisition may produce a cleaner image:

```text
Higher illumination
        ↓
More detected signal
        ↓
Cleaner microscopy image
```

However, increasing the illumination may also create unwanted biological and experimental effects:

```text
Higher illumination
        ↓
More photobleaching and phototoxicity
        ↓
Greater risk of disturbing long-term observation
```

A lower-exposure acquisition reduces the imaging burden on the sample, but the resulting image may be noisy:

```text
Lower illumination
        ↓
Less signal
        ↓
Noisier microscopy image
```

The central challenge of this notebook is therefore:

> **Can we train a denoising model using only noisy microscopy images, without using a clean image as the training target?**

Throughout the notebook, we will use the same microscopy experiment to study this question.



## From Supervised Denoising to Self-Supervision

The development of modern learning-based denoising can be understood as a gradual reduction in the amount of supervision required.

### Supervised restoration

A supervised denoiser is trained using paired noisy and clean images:

$$f_\theta(x_{\mathrm{noisy}}) \approx x_{\mathrm{clean}}$$

This can work well when reliable clean targets are available.

The difficulty is that clean biological reference images may be expensive, time-consuming, or physically difficult to acquire.



### Noise2Noise

Noise2Noise showed that a clean target is not always necessary.

Instead, the model can be trained using two independent noisy observations of the same underlying signal:

$$x_{\mathrm{noisy}}^{(1)} \rightarrow f_\theta \rightarrow x_{\mathrm{noisy}}^{(2)}$$

However, this still requires multiple observations of the same underlying scene.

For dynamic biological samples, repeated observations may not be perfectly aligned because cells, organelles, or other structures can move over time.



### Noise2Void

Noise2Void removes the need for paired observations.

The main idea is to hide selected pixels from the model and ask the model to predict them using surrounding spatial information.

A selected target pixel is temporarily removed or replaced:

```text
Neighboring pixels
        ↓
Predict the hidden center pixel
        ↓
Compare the prediction with the original noisy pixel
```

The model is therefore prevented from simply copying the noisy target pixel.

The training loss is evaluated only at selected blind-spot locations.

A simplified objective can be written as:

$$\mathcal{L}*{\mathrm{masked}} = \frac{\sum_i m_i\left(f*\theta(\tilde{x})_i-x_i\right)^2}{\sum_i m_i}$$

where:

* $x$ is the original noisy image;
* $\tilde{x}$ is the masked or modified input;
* $m_i$ indicates whether pixel $i$ is selected for training;
* $f_\theta(\tilde{x})_i$ is the predicted intensity at pixel $i$.

The important idea is that the model predicts selected pixels without directly observing their original values.


### Noise2Self

Noise2Self provides a more general explanation of this principle.

The prediction of one subset of pixels should not directly depend on the observed values of those same pixels.

This idea is often described using $\mathcal{J}$-invariance.

The intuition is:

```text
Biological structure
        ↓
Spatially predictable from neighboring context

Independent noise
        ↓
Difficult to predict from neighboring pixels
```

If the biological signal is spatially structured while the noise is sufficiently independent, a model can learn predictable image structure without directly learning the unpredictable noise.

---

## Why This Matters for Biological Imaging

Self-supervised denoising is especially relevant when:

* clean reference images are unavailable;
* repeated acquisition may damage the sample;
* the biological scene changes over time;
* only noisy experimental images are available;
* paired low-quality and high-quality acquisitions are difficult to align.

However, denoising also introduces an important risk.

A smoother image is not automatically a more biologically faithful image.

A denoising method may:

* remove weak but real structures;
* blur small biological boundaries;
* create artificial textures;
* suppress rare events;
* produce visually plausible but incorrect structures.

Therefore, this notebook will not evaluate denoising only by asking whether the image looks cleaner.

We will also ask:

> **Did the denoising process preserve the biological structures that were actually present?**



## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain why clean training targets can be difficult to obtain in biological imaging.

2. Compare supervised denoising, Noise2Noise, Noise2Void, and Noise2Self.

3. Explain the difference between image signal and image noise in a microscopy experiment.

4. Create a controlled noisy microscopy experiment for self-supervised denoising.

5. Compare classical filtering methods with learning-based denoising.

6. Explain the blind-spot principle used by Noise2Void.

7. Create masked training samples from noisy images.

8. Compute a loss only at selected masked pixel locations.

9. Explain the concept of $\mathcal{J}$-invariant prediction from Noise2Self.

10. Train a small neural network using only noisy microscopy data.

11. Compare noisy images, classical filtering results, and self-supervised denoising results.

12. Evaluate denoising using both image-quality metrics and biological structure preservation.

13. Explain when blind-spot denoising assumptions may fail.

14. Discuss why microscopy-specific imaging physics can matter for restoration.



## Tutorial Roadmap

This notebook is organized as follows:

```text
Part 1
Prepare a biological imaging experiment
        ↓
Task 1
Create the clean reference and noisy observation

Part 2
Understand the transition from supervised restoration
to self-supervised denoising
        ↓
Part 3
Compare classical denoising baselines
        ↓
Task 2
Analyze noise reduction and structure blurring

Part 4
Understand blind-spot self-supervision
        ↓
Task 3
Build masked training samples

Part 5
Learn from noisy images alone
        ↓
Task 4
Train a small self-supervised denoiser

Part 6
Evaluate biological structure preservation
        ↓
Part 7
Discuss limitations and microscopy-specific extensions
```



## Important Experimental Rule

During this tutorial, a clean reference image may be retained for evaluation.

However, the self-supervised denoising model must not use the clean reference as a training target.

The experimental separation is:

```text
Training:
Noisy microscopy image only

Evaluation:
Hidden clean reference
```

This allows us to answer two different questions:

1. Can the model learn without clean targets?

2. After training, how well did it preserve the underlying image structure?



## References

1. Weigert, M. et al.
   *Content-aware image restoration: pushing the limits of fluorescence microscopy.*
   Nature Methods, 2018.

2. Lehtinen, J. et al.
   *Noise2Noise: Learning Image Restoration without Clean Data.*
   ICML, 2018.

3. Krull, A., Buchholz, T.-O., and Jug, F.
   *Noise2Void: Learning Denoising from Single Noisy Images.*
   CVPR, 2019.

4. Batson, J. and Royer, L.
   *Noise2Self: Blind Denoising by Self-Supervision.*
   ICML, 2019.

5. Goncharova, A. et al.
   *Improving Blind Spot Denoising for Microscopy.*
   2020.


# Part 1 — Preparing the Biological Imaging Experiment

In this tutorial, we will use a fluorescence microscopy image of U2OS cell nuclei from the Broad Bioimage Benchmark Collection, BBBC039.

The image contains nuclei stained in the DNA channel and provides clear biological structures with different intensities, shapes, sizes, and local textures.

These properties make it useful for studying denoising because we can ask more than whether an image simply looks smoother.

We can also examine whether denoising:

* preserves weak nuclei;
* maintains nuclear boundaries;
* retains internal nuclear texture;
* removes real small structures;
* introduces artificial image features.



## A Controlled Low-Light Experiment

The central scenario of this tutorial is a simplified low-light fluorescence microscopy experiment.

We begin with a reference image:

$$x_{\mathrm{ref}}$$

We then simulate a lower-photon acquisition process.

The resulting noisy observation will be written as:

$$y$$

Our simplified acquisition pipeline is:

```text
Reference fluorescence image
        ↓
Reduced photon count
        ↓
Poisson shot noise
        ↓
Gaussian detector noise
        ↓
Noisy low-light observation
```

The noisy image can be described conceptually as:

$$y = \mathcal{P}(x_{\mathrm{ref}}) + n_{\mathrm{read}}$$

where:

* $\mathcal{P}(\cdot)$ represents signal-dependent Poisson shot noise;
* $n_{\mathrm{read}}$ represents Gaussian detector or readout noise.

The Poisson component is signal dependent. Brighter regions produce more photon counts but also different noise statistics.

The Gaussian component represents a simplified signal-independent electronic noise source.

---

## Important Experimental Separation

During this tutorial, the downloaded microscopy image will be stored as:

```text
clean_reference
```

The simulated low-light observation will be stored as:

```text
noisy_image
```

The self-supervised model will only use:

```text
noisy_image
```

during training.

The variable:

```text
clean_reference
```

will remain hidden from the training objective and will only be used later for evaluation.

The experimental rule is therefore:

```text
Training:
noisy_image only

Evaluation:
clean_reference revealed after training
```

This separation is essential.

If the clean reference were used as the training target, the experiment would become supervised denoising rather than Noise2Void-style self-supervised denoising.

---

## A Note About the Reference Image

The downloaded BBBC039 example image is treated as a controlled reference for this tutorial.

It should not be interpreted as a perfectly noise-free microscopy ground truth.

Our experiment therefore measures whether the model can recover the image before the additional simulated low-light corruption was applied.

This controlled design gives us a known reference for evaluation while still ensuring that the self-supervised model never uses that reference during training.


In [ ]:
# Part 1.1 — Environment setup
# ============================================================

import random
from io import BytesIO
from urllib.request import Request, urlopen

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch


# ------------------------------------------------------------
# Reproducibility
#
# We use a fixed random seed so that the simulated microscopy
# noise and later training experiments can be reproduced.
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Runtime device
#
# The notebook automatically uses a CUDA GPU when available.
# All core tasks are also designed to run on CPU.
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("Environment setup completed.")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)

In [ ]:
# Part 1.2 — Download the BBBC039 microscopy image
# ============================================================

BBBC039_IMAGE_URL = (
    "https://data.broadinstitute.org/bbbc/BBBC039/"
    "BBBC039exampleimage1.png"
)


def download_image_from_url(url):
    """
    Download an image from a public URL and return it as a PIL image.

    The image is loaded directly into memory, so no local image file
    needs to be stored inside the tutorial repository.
    """

    request = Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urlopen(request, timeout=30) as response:
        image_bytes = response.read()

    image = Image.open(
        BytesIO(image_bytes)
    )

    return image


# ------------------------------------------------------------
# Download the official BBBC039 example image.
# ------------------------------------------------------------

original_microscopy_image = download_image_from_url(
    BBBC039_IMAGE_URL
)


print("Image downloaded successfully.")
print("Original image mode:", original_microscopy_image.mode)
print("Original image size:", original_microscopy_image.size)


# ------------------------------------------------------------
# Display the downloaded image before preprocessing.
# ------------------------------------------------------------

plt.figure(figsize=(6, 5))

plt.imshow(
    original_microscopy_image,
    cmap="gray"
)

plt.title(
    "BBBC039 U2OS Cell Nuclei"
)

plt.axis("off")
plt.show()

## Task 1 — Prepare a Low-Light Microscopy Denoising Experiment

In this task, we will transform the downloaded BBBC039 microscopy image into a controlled low-light denoising experiment.

The goal is to create two variables:

```text
clean_reference
noisy_image
```

The first variable will represent the image before additional simulated low-light corruption.

The second variable will represent the noisy observation available to the self-supervised model.


### What You Need to Do

First, you will convert the downloaded image to grayscale.

Then, you will center-crop and resize the image to:

$$256 \times 256$$

The processed image will be converted into a PyTorch tensor with shape:

$$[B,C,H,W] = [1,1,256,256]$$

and intensity range:

$$[0,1]$$

This tensor will be stored as:

```text
clean_reference
```

Next, you will simulate a simplified low-light fluorescence acquisition.

---

### Step 1 — Simulate Photon Shot Noise

In fluorescence microscopy, photon detection is a counting process.

When fewer photons are detected, the relative uncertainty becomes stronger.

We model this using a Poisson process.

The clean image intensity is first converted into an expected photon count:

$$\lambda = Lx_{\mathrm{ref}}$$

where $L$ controls the simulated photon level.

We then sample:

$$k \sim \mathrm{Poisson}(\lambda)$$

and convert the counts back to an image:

$$x_{\mathrm{Poisson}} = \frac{k}{L}$$

A lower value of $L$ represents a more photon-limited acquisition.



### Step 2 — Add Detector Readout Noise

We then add a small Gaussian noise component:

$$n_{\mathrm{read}} \sim \mathcal{N}(0,\sigma^2)$$

The final observation is:

$$y = \mathrm{clip}\left(x_{\mathrm{Poisson}} + n_{\mathrm{read}},0,1\right)$$

This final tensor will be stored as:

```text
noisy_image
```



### Experimental Rule

The self-supervised model used later in the tutorial will receive only:

```text
noisy_image
```

The model must never use:

```text
clean_reference
```

as a training target.

The reference image will remain hidden until the evaluation stage.



### Expected Results

After completing this task, you should obtain:

```text
clean_reference.shape = [1, 1, 256, 256]

noisy_image.shape = [1, 1, 256, 256]
```

Both tensors should contain values in:

$$[0,1]$$

The noisy image should preserve the main nuclear structures but contain visibly stronger intensity fluctuations.

Bright nuclei should remain recognizable.

Dim structures and fine internal textures should become more difficult to distinguish.

This gives us the controlled denoising problem used throughout the rest of the tutorial.


In [ ]:
# Task 1 — Prepare a Low-Light Microscopy Denoising Experiment
# ============================================================
#
# Goal:
# Create two tensors:
#
# clean_reference -> evaluation only
# noisy_image     -> used for self-supervised training
#
# The model must never use clean_reference as a training target.


IMAGE_SIZE = 256
PHOTON_LEVEL = 20.0
GAUSSIAN_STD = 0.03


# ------------------------------------------------------------
# Step 1: Center-crop the microscopy image into a square.
# This avoids distorting nuclei during resizing.
# ------------------------------------------------------------

def center_square_crop(image):
    width, height = image.size
    crop_size = min(width, height)

    left = (width - crop_size) // 2
    top = (height - crop_size) // 2

    return image.crop((left, top, left + crop_size, top + crop_size))


# ------------------------------------------------------------
# Step 2: Convert a grayscale PIL image into a tensor.
#
# [H, W] -> [1, 1, H, W]
# intensity range -> [0, 1]
# ------------------------------------------------------------

def pil_to_tensor_01(image):
    image_array = np.asarray(image, dtype=np.float32)

    # WriteYourCodeHere
    image_array = image_array / 255.0

    # WriteYourCodeHere
    tensor = torch.from_numpy(image_array).float()

    return tensor.unsqueeze(0).unsqueeze(0)


# ------------------------------------------------------------
# Step 3: Prepare the reference image.
# ------------------------------------------------------------

grayscale_image = original_microscopy_image.convert("L")
cropped_image = center_square_crop(grayscale_image)

resize_method = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC
resized_image = cropped_image.resize((IMAGE_SIZE, IMAGE_SIZE), resample=resize_method)

# WriteYourCodeHere
clean_reference = pil_to_tensor_01(resized_image).to(device)


# ------------------------------------------------------------
# Step 4: Simulate a simplified low-light acquisition.
#
# Poisson noise models photon-counting uncertainty.
# Gaussian noise models detector readout noise.
# ------------------------------------------------------------

def simulate_low_light_acquisition(reference_image, photon_level=20.0, gaussian_std=0.03):

    # Convert normalized intensity into expected photon counts.
    # WriteYourCodeHere
    expected_counts = reference_image * photon_level

    # Sample photon counts and convert back to [0, 1]-like intensity values.
    # WriteYourCodeHere
    poisson_image = torch.poisson(expected_counts) / photon_level

    # Add Gaussian detector noise.
    # WriteYourCodeHere
    read_noise = gaussian_std * torch.randn_like(reference_image)

    # WriteYourCodeHere
    noisy_observation = torch.clamp(poisson_image + read_noise, 0.0, 1.0)

    return noisy_observation, poisson_image


# ------------------------------------------------------------
# Step 5: Create the noisy low-light observation.
# ------------------------------------------------------------

# WriteYourCodeHere
noisy_image, poisson_image = simulate_low_light_acquisition(
    clean_reference,
    photon_level=PHOTON_LEVEL,
    gaussian_std=GAUSSIAN_STD
)


# ------------------------------------------------------------
# Checkpoint
# ------------------------------------------------------------

expected_shape = (1, 1, IMAGE_SIZE, IMAGE_SIZE)

assert clean_reference.shape == expected_shape
assert noisy_image.shape == expected_shape
assert 0.0 <= clean_reference.min() and clean_reference.max() <= 1.0
assert 0.0 <= noisy_image.min() and noisy_image.max() <= 1.0


# ------------------------------------------------------------
# Measure the initial corruption level.
# ------------------------------------------------------------

initial_mse = torch.mean((noisy_image - clean_reference) ** 2).item()

print("Low-light microscopy experiment created successfully.")
print("clean_reference shape:", clean_reference.shape)
print("noisy_image shape:", noisy_image.shape)
print("Photon level:", PHOTON_LEVEL)
print("Gaussian noise std:", GAUSSIAN_STD)
print(f"Initial noisy-reference MSE: {initial_mse:.6f}")


# ------------------------------------------------------------
# Visualize the acquisition process.
# ------------------------------------------------------------

reference_display = clean_reference.squeeze().cpu().numpy()
poisson_display = poisson_image.clamp(0, 1).squeeze().cpu().numpy()
noisy_display = noisy_image.squeeze().cpu().numpy()
difference_display = torch.abs(noisy_image - clean_reference).squeeze().cpu().numpy()

images = [
    reference_display,
    poisson_display,
    noisy_display,
    difference_display
]

titles = [
    "Reference Image",
    "Poisson Shot Noise",
    "Low-Light Observation",
    "Absolute Difference"
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for axis, image, title in zip(axes, images, titles):
    axis.imshow(image, cmap="gray")
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()


print("Task 1 completed.")
print("Training data: noisy_image")
print("Evaluation only: clean_reference")

# Part 2 — From Supervised Restoration to Self-Supervision

In Task 1, we created a controlled microscopy experiment with two images:

```text
clean_reference
noisy_image
```

A conventional supervised denoising model would use them as an input-target pair:

$$f_\theta(\mathrm{noisy\_image}) \approx \mathrm{clean\_reference}$$

This is useful when clean targets are available.

However, the central question of this tutorial is different:

> **Can we train a denoiser when the clean reference is unavailable?**

To understand why Noise2Void and Noise2Self are important, it is useful to compare several denoising settings.

---

## 2.1 Supervised Restoration

In supervised restoration, the model receives a noisy image as input and a clean image as the training target:

$$x_{\mathrm{noisy}} \rightarrow f_\theta \rightarrow x_{\mathrm{clean}}$$

The model parameters are optimized by minimizing a loss such as:

$$\mathcal{L}_{\mathrm{supervised}} = \|f_\theta(x_{\mathrm{noisy}}) - x_{\mathrm{clean}}\|^2$$

This approach can be highly effective, but it requires paired noisy and clean images.

In biological imaging, such pairs may be difficult to obtain.

A cleaner image may require:

- longer exposure;
- stronger illumination;
- repeated acquisition;
- slower imaging.

These changes may increase photobleaching, phototoxicity, or motion between acquisitions.

The target image may therefore be expensive, imperfectly aligned, or unavailable.


## 2.2 Noise2Noise: Removing the Clean Target

Noise2Noise showed that a clean training target is not always necessary.

Suppose two noisy observations contain the same underlying signal:

$$y^{(1)} = x + n^{(1)}$$

$$y^{(2)} = x + n^{(2)}$$

The model can be trained using:

$$y^{(1)} \rightarrow f_\theta \rightarrow y^{(2)}$$

instead of a clean target.

The training objective becomes:

$$\mathcal{L}_{\mathrm{N2N}} = \|f_\theta(y^{(1)}) - y^{(2)}\|^2$$

If the two noise realizations are independent and centered around the same underlying signal, the model can still learn the shared image structure.

This removes the need for clean targets.

However, it introduces another requirement:

> **Two independent noisy observations of the same underlying scene are still needed.**

For living cells or moving biological structures, repeated acquisitions may not contain exactly the same scene.


## 2.3 Noise2Void: Learning from Noisy Images Alone

Noise2Void removes the need for paired observations.

Instead of using a second image as the target, the model creates a training problem inside the noisy image itself.

Selected pixels are hidden or replaced before the image is given to the network.

The model must then predict those pixels from their surrounding context.

The training process is:

```text
Noisy image
    ↓
Select target pixels
    ↓
Hide or replace those pixels
    ↓
Predict them from neighboring context
    ↓
Compute loss only at selected locations
```

Let:

- $y$ be the original noisy image;
- $\tilde{y}$ be the modified input;
- $m_i$ indicate whether pixel $i$ was selected.

The masked loss is:

$$\mathcal{L}_{\mathrm{masked}} = \frac{\sum_i m_i(f_\theta(\tilde{y})_i-y_i)^2}{\sum_i m_i}$$

The important point is that the model cannot directly copy the original value of a selected target pixel.

It must use information from neighboring pixels.


## 2.4 Why Can This Work?

The method relies on a difference between biological structure and noise.

Biological structures are often spatially correlated.

For example, neighboring pixels may belong to the same:

- nucleus;
- membrane;
- organelle;
- tissue boundary.

This means that the signal at one location may be partially predictable from nearby image content.

Independent noise behaves differently.

If a noise value at one pixel is independent of its neighbors, then the surrounding pixels cannot reliably predict that noise realization.

The model therefore has an incentive to learn:

```text
Predictable spatial structure
```

rather than:

```text
Unpredictable pixel-wise noise
```

This intuition is central to Noise2Void.


## 2.5 Noise2Self and $\mathcal{J}$-Invariance

Noise2Self provides a more general framework for the same principle.

Suppose the image coordinates are divided into subsets collected in $\mathcal{J}$.

A predictor is called $\mathcal{J}$-invariant when the prediction of one subset does not directly depend on the observed values of that same subset.

Conceptually:

```text
Predict pixel group J
        ↓
Do not use the observed values inside J
        ↓
Use information outside J
```

For a selected pixel $i$, this means:

$$f_\theta(y)_i \text{ must not directly depend on } y_i$$

Noise2Void implements this idea through masked or modified pixels.

Noise2Self explains the broader theoretical principle behind this type of self-supervision.

For this tutorial, the distinction is:

```text
Noise2Void
→ practical blind-spot training strategy

Noise2Self
→ general explanation of why self-supervised denoising can work
```


## 2.6 Comparing the Training Settings

| Method | Input | Training Target | Clean Target Required? | Paired Observation Required? |
|---|---|---|---|---|
| Supervised denoising | Noisy image | Clean image | Yes | Yes |
| Noise2Noise | Noisy image A | Noisy image B | No | Yes |
| Noise2Void | Modified noisy image | Selected noisy pixels | No | No |
| Noise2Self | Noisy data | Self-supervised prediction target | No | No |

The progression is therefore:

```text
Clean targets
    ↓
No clean targets, but paired noisy images
    ↓
Single noisy dataset
```



## 2.7 The Experimental Rule in This Notebook

In our controlled experiment, `clean_reference` exists only because we created the noisy observation ourselves.

However, the self-supervised model will behave as if the clean reference does not exist.

During training:

```text
Allowed:
noisy_image

Not allowed:
clean_reference
```

Only after training will we reveal `clean_reference` for evaluation.

This separation allows us to study two questions independently:

1. Can the model learn from noisy images alone?

2. How close is the final result to the image before additional corruption?


## 2.8 What Comes Next?

Before training a neural network, we will first establish simple classical denoising baselines.

In the next part, we will compare the noisy microscopy image with basic filtering methods and ask:

> **Does reducing visible noise always preserve biological structure?**

# Part 3 — Noise Behavior and Classical Denoising Baselines

Before training a self-supervised neural network, we first establish two simple classical denoising baselines.

Classical filters do not learn from data.

Instead, they apply fixed local operations to reduce image fluctuations.

In this tutorial, we compare:

- Gaussian filtering;
- median filtering.

The goal is not to study these methods in depth. Basic image filtering has already been introduced in earlier tutorials.

Here, they serve as baselines for one important question:

> **Does reducing visible noise automatically preserve biological structure?**

---

## 3.1 Gaussian Filtering

A Gaussian filter replaces each pixel with a weighted average of nearby pixels.

Pixels closer to the center receive larger weights.

The two-dimensional Gaussian kernel is:

$$G_\sigma(x,y)=\frac{1}{2\pi\sigma^2}\exp\left(-\frac{x^2+y^2}{2\sigma^2}\right)$$

The parameter $\sigma$ controls the smoothing strength.

A larger $\sigma$ produces stronger smoothing.

Gaussian filtering can reduce high-frequency fluctuations, but it may also blur:

- nuclear boundaries;
- weak structures;
- fine internal texture.

The central trade-off is:

```text
Stronger smoothing
        ↓
Less visible noise
        ↓
Greater risk of blurring real structure
```



## 3.2 Median Filtering

A median filter replaces each pixel with the median value inside a local neighborhood:

$$\hat{x}_i=\operatorname{median}\{y_j:j\in\mathcal{N}(i)\}$$

Unlike Gaussian filtering, the median is not a weighted average.

This makes median filtering particularly useful for isolated extreme values and impulse-like noise.

It may preserve sharp boundaries better than a strong averaging filter.

However, it can still modify:

- small structures;
- thin boundaries;
- local biological texture.



## 3.3 Why These Baselines Matter

A denoised image may look visually smoother while losing biologically meaningful information.

For example, a method may remove:

```text
Random noise
```

but it may also remove:

```text
Weak nuclei
Small boundaries
Fine intracellular texture
```

This means that visual smoothness alone is not enough to judge denoising quality.

In this part, we will compare the filtered images and inspect the image content removed by each method.

The clean reference remains hidden.

We are not yet asking:

> Which method is closest to the reference?

Instead, we are asking:

> What changes when a classical filter makes the image look cleaner?

## Task 2 — Compare Classical Denoising Baselines

In this task, you will apply two classical denoising methods to the low-light microscopy image created in Task 1.

You will compare:

```text
Noisy image
Gaussian filtering
Median filtering
```

The self-supervised model has not been introduced yet.

This task provides simple baselines that we will compare with the learned denoiser later.



### What You Need to Do

First, apply a Gaussian filter with:

$$\sigma=1.0$$

Then, apply a median filter with a:

$$3\times3$$

neighborhood.

You will inspect:

1. the complete microscopy image;
2. the same zoomed region for all methods;
3. the image content removed by each filter.

The removed content is calculated as:

$$r=|y-\hat{x}|$$

where $y$ is the noisy image and $\hat{x}$ is the filtered result.

This residual does not tell us whether the removed content is good or bad.

Instead, it helps us inspect an important question:

> **Did the filter remove random fluctuations, or did it also remove coherent biological structure?**



### Experimental Rule

The variable:

```text
clean_reference
```

must remain unused in this task.

The classical methods receive only:

```text
noisy_image
```

This keeps the comparison consistent with the noisy-only setting used later for self-supervised learning.



### Expected Results

The Gaussian filter should make the image smoother, but some nuclear boundaries and local texture may become softer.

The median filter may preserve some edges more strongly, but it can still alter small structures and local intensity patterns.

The residual images may contain more than random noise.

If recognizable boundaries or coherent structures appear in the residual, this suggests that the filter removed part of the biological signal as well.

In [ ]:
# Task 2 — Compare Classical Denoising Baselines
# ============================================================
#
# Goal:
# Compare Gaussian and median filtering using only noisy_image.
#
# clean_reference remains hidden in this task.


from scipy.ndimage import gaussian_filter, median_filter


GAUSSIAN_SIGMA = 1.0
MEDIAN_SIZE = 3
ZOOM_SIZE = 96


# ------------------------------------------------------------
# Step 1: Convert the noisy tensor into a NumPy image.
#
# Classical SciPy filters operate directly on NumPy arrays.
# ------------------------------------------------------------

noisy_array = noisy_image.squeeze().detach().cpu().numpy()


# ------------------------------------------------------------
# Step 2: Apply Gaussian filtering.
#
# Gaussian filtering reduces local high-frequency fluctuations
# through weighted spatial averaging.
# ------------------------------------------------------------

# WriteYourCodeHere
gaussian_array = gaussian_filter(noisy_array, sigma=GAUSSIAN_SIGMA)


# ------------------------------------------------------------
# Step 3: Apply median filtering.
#
# Each pixel is replaced by the median value inside a local
# neighborhood.
# ------------------------------------------------------------

# WriteYourCodeHere
median_array = median_filter(noisy_array, size=MEDIAN_SIZE)


# ------------------------------------------------------------
# Step 4: Convert both results back into PyTorch tensors.
#
# We keep the same [1, 1, H, W] shape as noisy_image so that all
# later methods can be compared consistently.
# ------------------------------------------------------------

def image_to_tensor(image):
    image = np.asarray(image, dtype=np.float32)
    return torch.from_numpy(image).unsqueeze(0).unsqueeze(0).to(device)


# WriteYourCodeHere
gaussian_denoised = image_to_tensor(np.clip(gaussian_array, 0.0, 1.0))

# WriteYourCodeHere
median_denoised = image_to_tensor(np.clip(median_array, 0.0, 1.0))


# ------------------------------------------------------------
# Checkpoint
# ------------------------------------------------------------

assert gaussian_denoised.shape == noisy_image.shape
assert median_denoised.shape == noisy_image.shape
assert torch.isfinite(gaussian_denoised).all()
assert torch.isfinite(median_denoised).all()


print("Classical denoising baselines created successfully.")
print("Gaussian sigma:", GAUSSIAN_SIGMA)
print("Median filter size:", MEDIAN_SIZE)


# ------------------------------------------------------------
# Step 5: Prepare images for visualization.
# ------------------------------------------------------------

noisy_display = noisy_image.squeeze().detach().cpu().numpy()
gaussian_display = gaussian_denoised.squeeze().detach().cpu().numpy()
median_display = median_denoised.squeeze().detach().cpu().numpy()

images = [noisy_display, gaussian_display, median_display]
titles = ["Noisy Image", "Gaussian Filter", "Median Filter"]


# ------------------------------------------------------------
# Step 6: Define one shared zoomed region.
#
# The same crop is used for all methods so that boundaries and
# local texture can be compared directly.
# ------------------------------------------------------------

zoom_start = (IMAGE_SIZE - ZOOM_SIZE) // 2
zoom_end = zoom_start + ZOOM_SIZE


# ------------------------------------------------------------
# Step 7: Compare the full images and zoomed regions.
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for column, (image, title) in enumerate(zip(images, titles)):
    axes[0, column].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title(title)
    axes[0, column].axis("off")

    zoomed_image = image[zoom_start:zoom_end, zoom_start:zoom_end]
    axes[1, column].imshow(zoomed_image, cmap="gray", vmin=0, vmax=1)
    axes[1, column].set_title(f"{title} — Zoom")
    axes[1, column].axis("off")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Step 8: Inspect the content removed by each filter.
#
# A residual may contain random noise, biological structure,
# or both.
# ------------------------------------------------------------

# WriteYourCodeHere
gaussian_removed = torch.abs(noisy_image - gaussian_denoised).squeeze().cpu().numpy()

# WriteYourCodeHere
median_removed = torch.abs(noisy_image - median_denoised).squeeze().cpu().numpy()

residual_vmax = max(np.percentile(gaussian_removed, 99), np.percentile(median_removed, 99))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(gaussian_removed, cmap="gray", vmin=0, vmax=residual_vmax)
axes[0].set_title("Content Removed by Gaussian Filter")
axes[0].axis("off")

axes[1].imshow(median_removed, cmap="gray", vmin=0, vmax=residual_vmax)
axes[1].set_title("Content Removed by Median Filter")
axes[1].axis("off")

plt.tight_layout()
plt.show()


print("Task 2 completed.")
print("The clean reference was not used.")
print("Next, we will create self-supervised training targets from noisy_image alone.")

# Part 4 — Blind-Spot Self-Supervision

So far, we have only used fixed denoising rules.

Gaussian and median filters can reduce visible fluctuations, but they do not learn which image structures are predictable from the microscopy data.

We now move to the central idea of Noise2Void:

> **Create a prediction task inside the noisy image itself.**

The key challenge is that the target pixel must not remain visible to the model.

Otherwise, the network could simply copy the noisy input value and learn the identity function.

---

## 4.1 Why Direct Noisy-to-Noisy Training Fails

Suppose we train a model using the same noisy image as both input and target:

$$f_\theta(y)\approx y$$

The loss would be:

$$\mathcal{L}=\|f_\theta(y)-y\|^2$$

The easiest solution is simply:

$$f_\theta(y)=y$$

The model can copy every noisy pixel directly.

This does not require learning biological structure.

Therefore, we need to prevent the model from directly observing the pixels used as training targets.



## 4.2 Hide the Target Pixel

For a selected pixel $i$, we create a modified input image:

$$\tilde{y}$$

The original value $y_i$ is removed or replaced before the image is given to the network.

The model receives:

```text
Modified noisy image
```

and predicts:

```text
Original noisy value at the selected location
```

The training process is:

```text
Original noisy image
        ↓
Select target pixels
        ↓
Replace their input values
        ↓
Predict the hidden pixels
        ↓
Compute loss only at target locations
```

The important separation is:

```text
Model input:
Modified noisy image

Training target:
Original noisy values at selected locations
```

No clean image is required.



## 4.3 Why Replace Pixels Instead of Setting Them to Zero?

A simple option would be to replace every target pixel with zero.

However, this creates an artificial pattern.

The model may learn that:

```text
Zero-valued pixel
        ↓
This location is probably a training target
```

Instead, a Noise2Void-style strategy replaces the selected pixel with a value sampled from its local neighborhood.

For example:

```text
Nearby noisy pixel
        ↓
Replace selected center pixel
        ↓
Network predicts original center value
```

The replacement value still looks locally plausible, but the original target value is no longer directly visible.



## 4.4 Local Neighborhood Replacement

Consider a selected target pixel at location $i$.

We choose another pixel $j$ from a local neighborhood:

$$j\in\mathcal{N}(i),\qquad j\neq i$$

The modified input becomes:

$$\tilde{y}_i=y_j$$

while the training target remains:

$$y_i$$

The model must therefore use spatial context to estimate the original value.

In this tutorial, we will use a:

$$5\times5$$

neighborhood.

The center location itself is excluded from the replacement candidates.



## 4.5 Compute Loss Only at Selected Pixels

Most pixels are left unchanged.

Only a small fraction are selected as training targets.

Let $m_i$ be a binary mask:

$$m_i=\begin{cases}1,&\text{if pixel }i\text{ is selected}\\0,&\text{otherwise}\end{cases}$$

The masked loss is:

$$\mathcal{L}_{\mathrm{masked}}=\frac{\sum_i m_i(f_\theta(\tilde{y})_i-y_i)^2}{\sum_i m_i}$$

Pixels outside the mask do not contribute to the training objective.

This is essential.

The model may see most of the noisy image, but it is only optimized on locations where the original input value was hidden.



## 4.6 Connection to $\mathcal{J}$-Invariance

Noise2Self describes the broader principle using $\mathcal{J}$-invariance.

For a selected group of pixels $J$, the prediction should not directly depend on the observed values inside that same group.

Conceptually:

```text
Predict pixels in J
        ↓
Hide their original values
        ↓
Use information outside J
```

Our masking strategy creates a practical approximation of this principle.

For selected pixels, the network cannot directly copy the original observation.

It must rely on surrounding context.



## 4.7 Important Assumption

This method relies on an important statistical idea:

```text
Biological signal
→ spatially correlated

Pixel-wise noise
→ difficult to predict from neighbors
```

If nearby pixels contain useful information about the biological structure, the network can learn that structure.

However, if the noise itself is strongly correlated across neighboring pixels, the same strategy may also learn part of the noise.

We will return to this limitation later.



## 4.8 What We Will Build

In the next task, we will create:

```text
masked_input
target_mask
```

from:

```text
noisy_image
```

The original noisy image will provide the target values.

The clean reference will remain completely unused.

The output of this task will become the training mechanism used in Part 5.

## Task 3 — Build Blind-Spot Training Samples

In this task, you will create self-supervised training samples from `noisy_image` alone.

You will randomly select a small fraction of pixels as training targets.

For each selected pixel:

1. keep its original noisy value as the training target;
2. replace its input value with a randomly selected neighboring pixel;
3. record its location in a binary mask.

The result will be:

```text
noisy_image
        ↓
Blind-spot transformation
        ↓
masked_input + target_mask
```

---

### What You Need to Do

You will create a function:

```python
create_blind_spot_sample(...)
```

that returns:

```text
masked_input
target_mask
```

The mask will have the same shape as the image:

$$[1,1,H,W]$$

with:

```text
1 → selected training pixel
0 → ignored by the training loss
```

A small masking probability will be used so that most of the image remains unchanged.



### Neighborhood Replacement

Each selected pixel will be replaced by another pixel from a local:

$$5\times5$$

neighborhood.

The center position itself will not be used.

This means the model cannot directly see the original value of the selected target pixel.



### Masked Training Loss

Later, a neural network will produce a prediction:

$$\hat{y}=f_\theta(\tilde{y})$$

The loss will be evaluated only where the mask equals one:

$$\mathcal{L}_{\mathrm{masked}}=\frac{\sum_i m_i(\hat{y}_i-y_i)^2}{\sum_i m_i}$$

In this task, you will also implement this loss function so that it can be reused during training.



### Expected Results

After completing this task, you should obtain:

```text
noisy_image
masked_input
target_mask
```

The masked input should look almost identical to the original noisy image because only a small fraction of pixels are modified.

When zooming into a local region, the selected training locations should become visible.

The target mask should clearly show which pixels will contribute to the future self-supervised loss.

The clean reference must remain unused.

In [ ]:
# Task 3 — Build Blind-Spot Training Samples
# ============================================================
#
# Goal:
# Create self-supervised training samples from noisy_image only.
#
# masked_input -> network input
# target_mask  -> locations used by the training loss


MASK_PROBABILITY = 0.02
NEIGHBORHOOD_SIZE = 5


# ------------------------------------------------------------
# Step 1: Create a blind-spot sample.
#
# Selected pixels are replaced by randomly chosen values from
# nearby locations. Their original noisy values remain the target.
# ------------------------------------------------------------

def create_blind_spot_sample(noisy_tensor, mask_probability=0.02, neighborhood_size=5):
    masked_input = noisy_tensor.clone()
    target_mask = torch.rand_like(noisy_tensor) < mask_probability

    radius = neighborhood_size // 2

    # Avoid selecting border pixels because they do not have a
    # complete local neighborhood.
    target_mask[:, :, :radius, :] = False
    target_mask[:, :, -radius:, :] = False
    target_mask[:, :, :, :radius] = False
    target_mask[:, :, :, -radius:] = False

    selected_pixels = torch.nonzero(target_mask[0, 0], as_tuple=False)

    # Build all possible local offsets except the center position.
    offsets = [(dy, dx) for dy in range(-radius, radius + 1) for dx in range(-radius, radius + 1) if not (dy == 0 and dx == 0)]
    offsets = torch.tensor(offsets, device=noisy_tensor.device)

    # Randomly choose one neighboring offset for every target pixel.
    # WriteYourCodeHere
    selected_offsets = offsets[torch.randint(len(offsets), (len(selected_pixels),), device=noisy_tensor.device)]

    target_y = selected_pixels[:, 0]
    target_x = selected_pixels[:, 1]

    source_y = target_y + selected_offsets[:, 0]
    source_x = target_x + selected_offsets[:, 1]

    # Replace each target pixel with a neighboring noisy value.
    # WriteYourCodeHere
    masked_input[0, 0, target_y, target_x] = noisy_tensor[0, 0, source_y, source_x]

    return masked_input, target_mask.float()


# ------------------------------------------------------------
# Step 2: Create one blind-spot training sample.
# ------------------------------------------------------------

# WriteYourCodeHere
masked_input, target_mask = create_blind_spot_sample(
    noisy_image,
    mask_probability=MASK_PROBABILITY,
    neighborhood_size=NEIGHBORHOOD_SIZE
)


# ------------------------------------------------------------
# Step 3: Define the masked training loss.
#
# Only selected target pixels contribute to the loss.
# ------------------------------------------------------------

def masked_mse_loss(prediction, target, mask):
    # WriteYourCodeHere
    squared_error = (prediction - target) ** 2
    return (squared_error * mask).sum() / mask.sum().clamp_min(1.0)


# ------------------------------------------------------------
# Checkpoint
# ------------------------------------------------------------

assert masked_input.shape == noisy_image.shape
assert target_mask.shape == noisy_image.shape
assert target_mask.sum() > 0
assert torch.isfinite(masked_input).all()

num_target_pixels = int(target_mask.sum().item())
total_pixels = target_mask.numel()
actual_mask_ratio = num_target_pixels / total_pixels

print("Blind-spot sample created successfully.")
print("Selected target pixels:", num_target_pixels)
print(f"Actual mask ratio: {actual_mask_ratio:.4f}")
print("Neighborhood size:", NEIGHBORHOOD_SIZE)


# ------------------------------------------------------------
# Step 4: Prepare visualization.
# ------------------------------------------------------------

noisy_display = noisy_image.squeeze().detach().cpu().numpy()
masked_display = masked_input.squeeze().detach().cpu().numpy()
mask_display = target_mask.squeeze().detach().cpu().numpy()
changed_display = torch.abs(masked_input - noisy_image).squeeze().detach().cpu().numpy()


# ------------------------------------------------------------
# Step 5: Show the complete images.
# ------------------------------------------------------------

images = [noisy_display, masked_display, mask_display, changed_display]

titles = [
    "Original Noisy Image",
    "Blind-Spot Input",
    "Target Mask",
    "Modified Pixels"
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for axis, image, title in zip(axes, images, titles):
    axis.imshow(image, cmap="gray")
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Step 6: Inspect the same zoomed region.
#
# The full images may look almost identical because only a small
# fraction of pixels are modified.
# ------------------------------------------------------------

zoom_start = (IMAGE_SIZE - ZOOM_SIZE) // 2
zoom_end = zoom_start + ZOOM_SIZE

zoom_images = [
    noisy_display[zoom_start:zoom_end, zoom_start:zoom_end],
    masked_display[zoom_start:zoom_end, zoom_start:zoom_end],
    mask_display[zoom_start:zoom_end, zoom_start:zoom_end],
    changed_display[zoom_start:zoom_end, zoom_start:zoom_end]
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for axis, image, title in zip(axes, zoom_images, titles):
    axis.imshow(image, cmap="gray")
    axis.set_title(f"{title} — Zoom")
    axis.axis("off")

plt.tight_layout()
plt.show()


print("Task 3 completed.")
print("Training input: masked_input")
print("Training target: noisy_image at selected mask locations")
print("clean_reference was not used.")

# Part 5 — Learning to Denoise from Noisy Images Alone

We now have all the components needed for self-supervised denoising.

From Task 3, we can create:

```text
masked_input
target_mask
```

from:

```text
noisy_image
```

We also defined a loss that is evaluated only at selected blind-spot locations.

The next step is to train a neural network.

---

## 5.1 The Self-Supervised Training Loop

Each training step follows the same procedure:

```text
noisy_image
        ↓
Create a new random blind-spot mask
        ↓
Replace selected pixels with neighboring values
        ↓
masked_input
        ↓
Neural network
        ↓
prediction
        ↓
Compare with original noisy values
only at masked locations
```

The model never receives `clean_reference` during training.

For each training step, a new random mask is generated.

This means that different pixels become prediction targets over time.

The training objective is:

$$\mathcal{L}_{\mathrm{masked}}=\frac{\sum_i m_i(f_\theta(\tilde{y})_i-y_i)^2}{\sum_i m_i}$$

where:

- $y$ is the original noisy image;
- $\tilde{y}$ is the blind-spot input;
- $m_i$ identifies selected target pixels;
- $f_\theta$ is the denoising network.



## 5.2 A Small Convolutional Denoiser

For this tutorial, we use a small convolutional neural network.

The goal is not to reproduce a large production-scale Noise2Void model.

Instead, the network is intentionally small so that we can focus on the learning mechanism.

The model contains several convolutional layers:

```text
Input microscopy image
        ↓
Convolution + ReLU
        ↓
Convolution + ReLU
        ↓
Convolution + ReLU
        ↓
Output prediction
```

Each convolution increases the receptive field of the network.

This allows the prediction at one pixel to use information from a wider surrounding region.



## 5.3 Why Generate a New Mask Every Step?

If we always masked exactly the same pixels, the model would only learn from those locations.

Instead, every training step creates a new random mask.

Over many iterations:

```text
Different pixels
        ↓
Become temporary training targets
        ↓
Provide many self-supervised examples
```

Even though we only have one microscopy image, the changing masks create many different prediction tasks.



## 5.4 Self-Supervised Inference

After training, we still want each output pixel to be predicted without directly using its own noisy value.

Therefore, we will reconstruct the image using multiple blind-spot groups.

For each group:

1. select a subset of pixels;
2. hide their original values;
3. run the trained network;
4. keep only the predictions at those hidden locations.

The groups are then combined into one final image.

Conceptually:

```text
Mask group 1 → predict group 1
Mask group 2 → predict group 2
Mask group 3 → predict group 3
...
        ↓
Combine all predictions
        ↓
Self-supervised denoised image
```

This keeps the final prediction consistent with the blind-spot principle.



## 5.5 Experimental Rule

During both training and inference:

```text
Allowed:
noisy_image

Not allowed:
clean_reference
```

The clean reference remains hidden.

It will only be used in Part 6, after the self-supervised model has finished training.



## 5.6 What We Will Observe

During training, we will monitor the masked loss.

A decreasing loss suggests that the model is becoming better at predicting hidden noisy pixels from their surrounding context.

However, a lower training loss does not automatically prove that the final image is biologically faithful.

Therefore, in this part we will only inspect:

- the training loss;
- the noisy input;
- the self-supervised output;
- local visual differences.

Formal comparison with the hidden reference will be performed later.

## Task 4 — Train a Small Self-Supervised Denoiser

In this task, you will train a convolutional neural network using only `noisy_image`.

The model will never use `clean_reference`.

---

### What You Need to Do

First, create a small convolutional denoiser.

Then, repeat the following training procedure:

```text
Create a new blind-spot sample
        ↓
Predict the image
        ↓
Compute loss only at masked pixels
        ↓
Update the network
```

A different random mask will be generated at every training step.



### Training Data

The network input is:

```text
masked_input
```

The target values come from:

```text
noisy_image
```

but only where:

```text
target_mask = 1
```

The clean reference is not used.



### Final Reconstruction

After training, the image will be reconstructed using several blind-spot groups.

Each pixel will be predicted while its original noisy value is hidden.

The combined output will be stored as:

```text
self_supervised_denoised
```

This variable will be used in Part 6 for comparison with the classical baselines and the hidden reference.



### Expected Results

The training loss should generally decrease over time.

The final output should appear smoother than the original noisy image while preserving the major nuclear structures.

Small differences may be easier to observe in a zoomed region.

At this stage, do not decide which method is best.

The hidden clean reference has not yet been used.

In [ ]:
# Task 4 — Train a Small Self-Supervised Denoiser
# ============================================================
#
# Goal:
# Train a CNN using only noisy_image and blind-spot targets.
#
# clean_reference remains hidden throughout this task.


import torch.nn as nn


NUM_STEPS = 300
LEARNING_RATE = 1e-3
MASK_GROUP_SIZE = 5


# ------------------------------------------------------------
# Step 1: Define a small convolutional denoiser.
#
# Multiple 3 x 3 convolutions allow each prediction to use
# information from a wider local neighborhood.
# ------------------------------------------------------------

class SmallDenoiser(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=3, padding=1)
        )

    def forward(self, x):
        return self.network(x)


# WriteYourCodeHere
model = SmallDenoiser().to(device)

# WriteYourCodeHere
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


num_parameters = sum(parameter.numel() for parameter in model.parameters())

print("Model created successfully.")
print("Trainable parameters:", num_parameters)


# ------------------------------------------------------------
# Step 2: Train using a new blind-spot sample at every step.
# ------------------------------------------------------------

loss_history = []

model.train()

for step in range(NUM_STEPS):

    # Create a new random self-supervised training sample.
    masked_input, target_mask = create_blind_spot_sample(
        noisy_image,
        mask_probability=MASK_PROBABILITY,
        neighborhood_size=NEIGHBORHOOD_SIZE
    )

    # Predict the full image.
    # WriteYourCodeHere
    prediction = model(masked_input)

    # Compute loss only at hidden target locations.
    # WriteYourCodeHere
    loss = masked_mse_loss(prediction, noisy_image, target_mask)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if (step + 1) % 50 == 0:
        print(f"Step {step + 1:3d}/{NUM_STEPS} | Loss: {loss.item():.6f}")


print("Training completed.")


# ------------------------------------------------------------
# Step 3: Plot the training loss.
# ------------------------------------------------------------

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Training Step")
plt.ylabel("Masked MSE Loss")
plt.title("Self-Supervised Training Loss")
plt.grid(alpha=0.3)
plt.show()


# ------------------------------------------------------------
# Step 4: Create one masked input from a fixed target mask.
#
# This helper is used during blind-spot reconstruction.
# ------------------------------------------------------------

def apply_blind_spot_mask(noisy_tensor, target_mask, neighborhood_size=5):
    masked_input = noisy_tensor.clone()
    radius = neighborhood_size // 2

    selected_pixels = torch.nonzero(target_mask[0, 0], as_tuple=False)

    if len(selected_pixels) == 0:
        return masked_input

    offsets = [(dy, dx) for dy in range(-radius, radius + 1) for dx in range(-radius, radius + 1) if not (dy == 0 and dx == 0)]
    offsets = torch.tensor(offsets, device=noisy_tensor.device)

    selected_offsets = offsets[torch.randint(len(offsets), (len(selected_pixels),), device=noisy_tensor.device)]

    target_y = selected_pixels[:, 0]
    target_x = selected_pixels[:, 1]

    source_y = torch.clamp(target_y + selected_offsets[:, 0], 0, noisy_tensor.shape[-2] - 1)
    source_x = torch.clamp(target_x + selected_offsets[:, 1], 0, noisy_tensor.shape[-1] - 1)

    masked_input[0, 0, target_y, target_x] = noisy_tensor[0, 0, source_y, source_x]

    return masked_input


# ------------------------------------------------------------
# Step 5: Reconstruct the image using blind-spot groups.
#
# Every pixel is assigned to one group.
#
# When a group is predicted, its original pixel values are hidden.
# ------------------------------------------------------------

def blind_spot_reconstruct(model, noisy_tensor, group_size=5):
    reconstruction = torch.zeros_like(noisy_tensor)

    model.eval()

    with torch.no_grad():
        for row_group in range(group_size):
            for col_group in range(group_size):

                target_mask = torch.zeros_like(noisy_tensor, dtype=torch.bool)
                target_mask[:, :, row_group::group_size, col_group::group_size] = True

                masked_input = apply_blind_spot_mask(
                    noisy_tensor,
                    target_mask,
                    neighborhood_size=NEIGHBORHOOD_SIZE
                )

                prediction = model(masked_input)

                # WriteYourCodeHere
                reconstruction[target_mask] = prediction[target_mask]

    return torch.clamp(reconstruction, 0.0, 1.0)


# ------------------------------------------------------------
# Step 6: Create the final self-supervised denoised image.
# ------------------------------------------------------------

# WriteYourCodeHere
self_supervised_denoised = blind_spot_reconstruct(
    model,
    noisy_image,
    group_size=MASK_GROUP_SIZE
)


# ------------------------------------------------------------
# Checkpoint
# ------------------------------------------------------------

assert self_supervised_denoised.shape == noisy_image.shape
assert torch.isfinite(self_supervised_denoised).all()
assert self_supervised_denoised.min() >= 0.0
assert self_supervised_denoised.max() <= 1.0


print("Blind-spot reconstruction completed.")
print("Output shape:", self_supervised_denoised.shape)


# ------------------------------------------------------------
# Step 7: Compare the noisy input and self-supervised output.
#
# clean_reference is still not used.
# ------------------------------------------------------------

noisy_display = noisy_image.squeeze().detach().cpu().numpy()
self_supervised_display = self_supervised_denoised.squeeze().detach().cpu().numpy()

images = [noisy_display, self_supervised_display]
titles = ["Noisy Image", "Self-Supervised Denoising"]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for axis, image, title in zip(axes, images, titles):
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Step 8: Compare the same zoomed region.
# ------------------------------------------------------------

zoom_images = [
    noisy_display[zoom_start:zoom_end, zoom_start:zoom_end],
    self_supervised_display[zoom_start:zoom_end, zoom_start:zoom_end]
]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for axis, image, title in zip(axes, zoom_images, titles):
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"{title} — Zoom")
    axis.axis("off")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Step 9: Show what the model changed.
#
# This is only a visualization.
# It does not evaluate whether the changes are correct.
# ------------------------------------------------------------

model_change = torch.abs(noisy_image - self_supervised_denoised).squeeze().detach().cpu().numpy()

plt.figure(figsize=(5, 4))
plt.imshow(model_change, cmap="gray")
plt.title("Changes Introduced by the Self-Supervised Model")
plt.axis("off")
plt.show()


print("Task 4 completed.")
print("Training used noisy_image only.")
print("clean_reference remains hidden until Part 6.")

# Part 6 — Does Denoising Preserve Biological Structure?

The self-supervised model has now finished training.

For the first time in this tutorial, we can reveal:

```text
clean_reference
```

The reference image was never used as a training target.

It is now used only to evaluate the final denoising results.

We will compare:

```text
Noisy image
Gaussian filtering
Median filtering
Self-supervised denoising
```

The central question is:

> **Did the denoising method remove noise while preserving the biological structures present in the reference image?**



## 6.1 Why Visual Smoothness Is Not Enough

A denoised image may look cleaner because random fluctuations have been reduced.

However, the same method may also remove:

- weak nuclei;
- thin boundaries;
- fine internal texture;
- small biological structures.

A smoother result is therefore not automatically a better result.

We need to compare each method against the hidden reference.



## 6.2 Mean Absolute Error

The Mean Absolute Error measures the average pixel-wise difference between a reconstruction $\hat{x}$ and the reference $x$:

$$\mathrm{MAE}=\frac{1}{N}\sum_i|\hat{x}_i-x_i|$$

A lower MAE indicates that the reconstructed intensities are closer to the reference.

However, MAE does not directly describe whether the spatial structure is preserved.



## 6.3 Peak Signal-to-Noise Ratio

Peak Signal-to-Noise Ratio is derived from the Mean Squared Error:

$$\mathrm{PSNR}=10\log_{10}\left(\frac{L^2}{\mathrm{MSE}}\right)$$

where $L$ is the maximum possible image intensity.

Because our images are normalized to $[0,1]$:

$$L=1$$

A higher PSNR generally indicates smaller reconstruction error.



## 6.4 Structural Similarity

Structural Similarity Index Measure, or SSIM, compares local image properties such as:

- intensity;
- contrast;
- spatial structure.

Its range is approximately:

$$[-1,1]$$

with values closer to:

$$1$$

indicating stronger structural similarity.

For biological imaging, SSIM is particularly useful because two images may have similar average pixel errors while preserving boundaries and textures very differently.



## 6.5 Edge Preservation

Many important biological structures are defined by spatial intensity transitions.

For example:

- nuclear boundaries;
- membrane-like structures;
- small object borders.

We therefore also compare image gradients.

A denoising method that removes too much high-frequency information may produce smoother images but weaker boundaries.

The goal is not to preserve every noisy fluctuation.

The goal is to preserve coherent biological edges while suppressing unpredictable noise.



## 6.6 Evaluation Strategy

We will evaluate all four images using:

```text
MAE
PSNR
SSIM
```

We will then inspect:

```text
Full images
Zoomed regions
Error maps
Edge maps
```

No single metric is sufficient on its own.

The final interpretation should consider both numerical accuracy and biological structure preservation.

In [ ]:
%pip install -q scikit-image

In [ ]:
# ============================================================
# Part 6 — Evaluate Biological Structure Preservation
# ============================================================
#
# The clean reference is revealed for the first time.
#
# We compare:
# noisy_image
# gaussian_denoised
# median_denoised
# self_supervised_denoised


from skimage.metrics import peak_signal_noise_ratio, structural_similarity


ZOOM_SIZE = 96


# ------------------------------------------------------------
# Step 1: Convert all images to NumPy arrays.
# ------------------------------------------------------------

reference_array = clean_reference.squeeze().detach().cpu().numpy()
noisy_array = noisy_image.squeeze().detach().cpu().numpy()
gaussian_array = gaussian_denoised.squeeze().detach().cpu().numpy()
median_array = median_denoised.squeeze().detach().cpu().numpy()
self_supervised_array = self_supervised_denoised.squeeze().detach().cpu().numpy()


results = {
    "Noisy": noisy_array,
    "Gaussian": gaussian_array,
    "Median": median_array,
    "Self-Supervised": self_supervised_array
}


# ------------------------------------------------------------
# Step 2: Compute MAE, PSNR, and SSIM.
# ------------------------------------------------------------

def evaluate_image(reference, prediction):
    mae = np.mean(np.abs(reference - prediction))
    psnr = peak_signal_noise_ratio(reference, prediction, data_range=1.0)
    ssim = structural_similarity(reference, prediction, data_range=1.0)
    return mae, psnr, ssim


metrics = {}

for name, image in results.items():
    # WriteYourCodeHere
    metrics[name] = evaluate_image(reference_array, image)


# ------------------------------------------------------------
# Step 3: Print the evaluation table.
#
# Lower MAE is better.
# Higher PSNR and SSIM are better.
# ------------------------------------------------------------

print("Denoising Evaluation")
print("-" * 58)
print(f"{'Method':<18}{'MAE':>10}{'PSNR':>12}{'SSIM':>12}")
print("-" * 58)

for name, (mae, psnr, ssim) in metrics.items():
    print(f"{name:<18}{mae:>10.4f}{psnr:>12.2f}{ssim:>12.4f}")


# ------------------------------------------------------------
# Step 4: Prepare the comparison images.
# ------------------------------------------------------------

comparison_images = [
    reference_array,
    noisy_array,
    gaussian_array,
    median_array,
    self_supervised_array
]

comparison_titles = [
    "Reference",
    "Noisy",
    "Gaussian",
    "Median",
    "Self-Supervised"
]


# ------------------------------------------------------------
# Step 5: Define one shared zoomed region.
#
# The same crop is used for every method.
# ------------------------------------------------------------

image_size = reference_array.shape[0]
zoom_start = (image_size - ZOOM_SIZE) // 2
zoom_end = zoom_start + ZOOM_SIZE


# ------------------------------------------------------------
# Step 6: Compare full images and zoomed regions.
#
# Top row:
# Complete microscopy images
#
# Bottom row:
# The same local region for every method
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 5, figsize=(18, 8))

for column, (image, title) in enumerate(zip(comparison_images, comparison_titles)):

    axes[0, column].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title(title)
    axes[0, column].axis("off")

    zoomed_image = image[zoom_start:zoom_end, zoom_start:zoom_end]

    axes[1, column].imshow(zoomed_image, cmap="gray", vmin=0, vmax=1)
    axes[1, column].set_title(f"{title} — Zoom")
    axes[1, column].axis("off")


plt.tight_layout()
plt.show()


print("Part 6 evaluation completed.")
print("The clean reference was used only after training.")

# Part 7 — Limitations and Microscopy-Specific Extensions

This tutorial demonstrated that a denoising model can be trained without using a clean image as the training target.

The complete learning process was:

```text
Noisy microscopy image
        ↓
Select hidden target pixels
        ↓
Replace them with neighboring values
        ↓
Predict the original noisy values
        ↓
Compute loss only at hidden locations
        ↓
Train from noisy data alone
```

The clean reference was used only after training for evaluation.

This experiment illustrates the central idea behind Noise2Void and the broader self-supervised principle discussed in Noise2Self.

However, successful noise reduction does not guarantee perfect biological restoration.

---

## 7.1 What Did the Model Actually Learn?

The model learned to predict hidden pixel values from nearby image context.

This works because biological structures are often spatially correlated.

For example, neighboring pixels may belong to the same:

- nucleus;
- membrane;
- organelle;
- tissue structure.

The model can therefore learn predictable spatial patterns from the noisy image.

At the same time, independent pixel-wise noise is difficult to predict from neighboring pixels.

The ideal behavior is:

```text
Predictable biological structure
        ↓
Preserved

Unpredictable pixel-wise noise
        ↓
Suppressed
```



## 7.2 Why Can Over-Smoothing Occur?

In our experiment, the self-supervised result may appear much smoother than the noisy image.

This indicates that the model successfully reduced visible fluctuations.

However, some fine image details may also become weaker.

The reason is that the model predicts the most likely value from surrounding context.

When a local structure is uncertain, minimizing Mean Squared Error often encourages predictions close to the local average.

This can produce:

```text
Less noise
        ↓
Smoother image
        ↓
Possible loss of fine structure
```

Therefore:

> **A cleaner-looking image is not automatically a more biologically faithful image.**

Over-smoothing is an important result to analyze, not simply a training failure.



## 7.3 The Main Assumptions Behind Blind-Spot Denoising

Blind-spot self-supervision works best when two important assumptions are approximately satisfied.

### Assumption 1 — The signal is spatially correlated

The underlying biological structure should be partly predictable from nearby pixels.

### Assumption 2 — The noise is difficult to predict from neighbors

The noise at one pixel should be approximately independent of the noise at nearby pixels.

Conceptually:

```text
Signal:
spatially predictable

Noise:
locally unpredictable
```

When these assumptions hold, the model can learn structure without learning the exact noise realization.



## 7.4 When Can These Assumptions Fail?

Real microscopy noise is not always independent.

Possible problems include:

- correlated detector noise;
- striping artifacts;
- fixed-pattern noise;
- reconstruction artifacts;
- spatially structured background;
- optical blur;
- repeated acquisition artifacts.

If the noise itself is spatially correlated, the model may begin to predict and preserve part of that noise.

The self-supervised assumption becomes less reliable.



## 7.5 Biological Risks

A denoising model may unintentionally:

- remove weak but real structures;
- blur thin boundaries;
- suppress rare events;
- erase small objects;
- create artificial smooth textures;
- alter quantitative intensity measurements.

This is especially important in biological imaging.

A visually plausible image may still be biologically incorrect.

For example, a weak nucleus or small fluorescent structure may be mistaken for noise and removed.

Therefore, denoising should not be evaluated only by appearance.



## 7.6 Limitations of This Tutorial

This notebook was designed to explain the learning mechanism, not to reproduce a full production-scale Noise2Void pipeline.

Several simplifications were made.

### Controlled reference image

The BBBC039 image was treated as a reference before additional simulated corruption.

It is not a perfectly noise-free microscopy ground truth.

### Simulated noise

We used a simplified Poisson-Gaussian noise model.

Real microscopy systems may contain more complex and spatially correlated noise.

### Single-image experiment

The model was trained on one microscopy image.

Real restoration systems often use larger image collections or many image patches.

### Small convolutional network

The network was intentionally compact so that the experiment could run quickly.

A larger architecture may capture more complex biological structures.

### Simplified blind-spot strategy

The masking and neighbor-replacement procedure illustrates the core mechanism.

It should not be interpreted as an exact reproduction of every detail in the original Noise2Void implementation.



## 7.7 Why Microscopy Physics Matters

Microscopy images are not arbitrary natural images.

The observed image is shaped by the imaging system itself.

Important physical factors include:

```text
Optical resolution
Point spread function
Photon statistics
Detector characteristics
Acquisition settings
```

A denoising method that ignores these factors may remove image content that is physically meaningful or create frequencies that the microscope could not realistically produce.

This motivates microscopy-specific extensions to blind-spot denoising.

For example, later work on improving blind-spot denoising for microscopy incorporates knowledge about the imaging system and its frequency behavior.

The broader lesson is:

> **A mathematically effective image denoiser is not automatically a physically correct microscopy restoration model.**



## 7.8 From Noise2Void to More Advanced Methods

This tutorial focused on the core self-supervised principle.

More advanced approaches extend this idea in several directions:

```text
Noise modeling
        ↓
Probabilistic prediction

Microscopy physics
        ↓
PSF-aware restoration

Structured noise
        ↓
Specialized blind-spot architectures

Larger datasets
        ↓
More robust representation learning
```

Examples include:

- Probabilistic Noise2Void;
- microscopy-specific blind-spot denoising;
- physics-informed restoration;
- restoration models that explicitly model the image formation process.



## 7.9 Final Takeaway

This tutorial started with a practical question:

> **Can we train a denoising model when clean microscopy targets are unavailable?**

The answer is yes, under suitable assumptions.

Noise2Void shows how noisy images can create their own training targets through blind-spot masking.

Noise2Self provides a broader explanation of why this type of self-supervision can work.

The complete workflow was:

```text
Prepare a noisy microscopy experiment
        ↓
Compare classical denoising baselines
        ↓
Create blind-spot training samples
        ↓
Train from noisy images alone
        ↓
Evaluate against a hidden reference
        ↓
Analyze structure preservation and failure modes
```

The most important lesson is not simply that self-supervised denoising can reduce noise.

It is that:

> **Noise reduction and biological fidelity are not the same objective.**

A successful microscopy restoration method must reduce unwanted noise while preserving the structures and measurements that matter biologically.

---

## References

1. Weigert, M. et al.  
   *Content-aware image restoration: pushing the limits of fluorescence microscopy.*  
   Nature Methods, 2018.

2. Lehtinen, J. et al.  
   *Noise2Noise: Learning Image Restoration without Clean Data.*  
   ICML, 2018.

3. Krull, A., Buchholz, T.-O., and Jug, F.  
   *Noise2Void: Learning Denoising from Single Noisy Images.*  
   CVPR, 2019.

4. Batson, J. and Royer, L.  
   *Noise2Self: Blind Denoising by Self-Supervision.*  
   ICML, 2019.

5. Goncharova, A. et al.  
   *Improving Blind Spot Denoising for Microscopy.*  
   2020.



## Further Reading

- Probabilistic Noise2Void
- Physics-informed microscopy restoration
- Self-supervised denoising for correlated noise
- Real fluorescence microscopy denoising benchmarks